# Preliminary Correlation & Causality Analysis (Phase 1)

This notebook applies our core causality tools to the *entire raw timeseries* of each session.
No advanced preprocessing (beyond basic cleaning) is applied yet.

**Methods**:
1.  **Pearson Correlation**: Baseline linear relationship.
2.  **CCM**: Nonlinear causality (Sugihara).
3.  **Transfer Entropy (Gaussian)**: Information flow.
4.  **VarLiNGAM**: Multivariate causal discovery.

**Config**: Analyzing the MIDDLE 2,000 frames of the first 2 sessions for a quick smoke test.

**Goal**: Identify strong candidate pairs and potential directionality in the raw data.

In [1]:
import sys
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

# Add project root to path
project_root = Path("../../").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from monoamine_interactions.analysis.phase_1_causality import utils, ccm, te, varlingam

## 1. Load Data

In [2]:
# Update path to point correctly inside the project folder
pkl_path = project_root / "monoamine_interactions" / "data" / "sessions_jonathan_juvenile_intruder_dirty.pkl"

print(f"Loading {pkl_path}...")
with open(pkl_path, "rb") as f:
    sessions = pickle.load(f)

print(f"Loaded {len(sessions)} sessions.")

Loading /home/fsp585/Desktop/FiberphotometryCode/monoamine_interactions/data/sessions_jonathan_juvenile_intruder_dirty.pkl...
Loaded 12 sessions.


## 2. Analysis Config

In [3]:
results = []

# Params
MAX_FRAMES = 50000 # Limit for speed in prelim test
LAG = 5  # For TE/CCM/LiNGAM
EMBED = 3

## 3. Run Pairwise Analysis Loop

In [4]:
for s in tqdm(sessions): # LIMIT TO 2 SESSIONS
    # Identify signals
    region_map = {k: v for k, v in s.metadata.items() 
                  if str(k).startswith("signal_") and isinstance(v, str)}
    regions = list(region_map.values())
    
    if len(regions) < 2:
        continue

    # Run all pairs
    from utils.photometry import motion_correct_biexponential
    for i, r_src in enumerate(regions):
        for j, r_tgt in enumerate(regions):
            if i == j: continue
            
            try:
                col_src = next((k for k, v in region_map.items() if v == r_src))
                col_tgt = next((k for k, v in region_map.items() if v == r_tgt))

                # Extract raw 470 traces AND 415 Reference traces
                src_raw = s.photometry['470'][col_src].values
                src_iso = s.photometry['415'][col_src].values
                
                tgt_raw = s.photometry['470'][col_tgt].values
                tgt_iso = s.photometry['415'][col_tgt].values

                # Apply Double Exponential dF/F fit + Linear Regression Isosbestic Correction
                src_dff = motion_correct_biexponential(src_raw, src_iso)
                tgt_dff = motion_correct_biexponential(tgt_raw, tgt_iso)

                # Ignore exclusions by dropping NaNs (which were set by the pre-exclusions)
                valid = ~np.isnan(src_dff) & ~np.isnan(tgt_dff)
                src = src_dff[valid]
                tgt = tgt_dff[valid]
                
                if len(src) < 100: continue

                # Slice MIDDLE segment
                if len(src) > MAX_FRAMES:
                    start = (len(src) - MAX_FRAMES) // 2
                    end = start + MAX_FRAMES
                    src = src[start:end]
                    tgt = tgt[start:end]
                
                # 1. Pearson
                pearson_r = np.corrcoef(src, tgt)[0, 1]
                
                # 2. TE
                te_val = te.compute_transfer_entropy_gaussian(src, tgt, lag=LAG)
                
                # 3. CCM
                r_s_x_t, r_t_x_s, _ = ccm.run_ccm_analysis(src, tgt, embed_dim=EMBED, lag=LAG)
                
                # Store
                results.append({
                    "session": s.session_id,
                    "animal": s.metadata.get("animal_id", "?"),
                    "source": r_src,
                    "target": r_tgt,
                    "pearson_r": pearson_r,
                    "te_bits": te_val,
                    "ccm_cause_skill": r_t_x_s 
                })
            except Exception as e:
                pass # Skip errors for prelim

print("Done.")

100%|██████████| 12/12 [00:51<00:00,  4.29s/it]

Done.


## 4. Multivariate Analysis (LiNGAM)

In [5]:
lingam_results = []

for s in tqdm(sessions): # LIMIT TO 2 SESSIONS
    region_map = {k: v for k, v in s.metadata.items() 
                  if str(k).startswith("signal_") and isinstance(v, str)}
    regions = list(region_map.values())
    
    if len(regions) < 2: continue
    
    try:
        from utils.photometry import motion_correct_biexponential
        
        # Build global mask across all regions to ensure parallel timeseries
        global_mask = None
        dff_dict = {}
        for r in regions:
            col = next((k for k, v in region_map.items() if v == r))
            raw_470 = s.photometry['470'][col].values
            raw_415 = s.photometry['415'][col].values
            
            dff = motion_correct_biexponential(raw_470, raw_415)
            dff_dict[r] = dff
            
            mask = ~np.isnan(dff)
            if global_mask is None:
                global_mask = mask
            else:
                global_mask = global_mask & mask
                
        # Build DF from the tightly masked segments
        data_dict = {}
        valid_regs = []
        
        for r in regions:
            tr = dff_dict[r][global_mask]
            
            total_len = len(tr)
            start = 0
            end = total_len
            if total_len > MAX_FRAMES:
                start = (total_len - MAX_FRAMES) // 2
                end = start + MAX_FRAMES
                
            data_dict[r] = tr[start:end]
            valid_regs.append(r)
            
        df_multi = pd.DataFrame(data_dict)
        
        res = varlingam.run_varlingam(df_multi, lags=LAG)
        adj = res['adjacency_matrices'][0] # Check lag 0 (instantaneous)
        
        lingam_results.append({
            "session": s.session_id,
            "regions": valid_regs,
            "adj_matrix": adj
        })

    except Exception:
        pass

100%|██████████| 12/12 [00:45<00:00,  3.77s/it]


## 5. Review Results

In [6]:
df_res = pd.DataFrame(results)
if not df_res.empty:
    # Aggregate by pair type across all animals
    agg = df_res.groupby(["source", "target"])[["ccm_cause_skill", "pearson_r", "te_bits"]].mean()
    print("\n--- Aggregated Causality Scores (All Animals) ---")
    print(agg.sort_values("ccm_cause_skill", ascending=False))
else:
    print("No results generate.")


--- Aggregated Causality Scores (All Animals) ---
                   ccm_cause_skill  pearson_r   te_bits
source   target                                        
lNAc_DA  rNAc_5HT         0.675094  -0.068841  0.000029
lNAc_5HT rNAc_DA          0.604121   0.414876  0.004972
rNAc_DA  lNAc_5HT         0.315274   0.414876  0.000942
NAc_5HT  DRN_NE           0.273203   0.013907  0.000892
rNAc_5HT lNAc_DA          0.120911  -0.068841  0.001433
DRN_NE   NAc_5HT         -0.017160   0.013907  0.001011


In [7]:
# ==========================================
# Phase 4: Multivariate Network Discovery (LiNGAM)
# ==========================================
import numpy as np
import pandas as pd
from tqdm import tqdm
# Assuming varlingam is imported from your utils:
# from monoamine_interactions.analysis.phase_1_causality import varlingam

lingam_results = []
LAG = 5
MAX_FRAMES = 50000 

print("Running LiNGAM across sessions...")
for s in tqdm(sessions):
    # 1. Identify signals
    region_map = {k: v for k, v in s.metadata.items() 
                  if str(k).startswith("signal_") and isinstance(v, str)}
    regions = list(region_map.values())
    
    if len(regions) < 2: continue
    
    try:
        data_dict = {}
        valid_regs = []
        
        # 2. Extract your ISOSBESTIC-CORRECTED traces here
        # (Replace this block with whatever function you used to get your clean iso-traces)
        for r in regions:
            col = next((k for k, v in region_map.items() if v == r))
            
            # --- YOUR ISO CORRECTION LOGIC HERE ---
            # e.g., clean_trace = compute_iso_corrected_dff(raw_470, raw_iso)
            # For now, we assume 'clean_trace' is your final artifact-free array
            clean_trace = s.photometry['470'][col].values # Placeholder
            # --------------------------------------
            
            total_len = len(clean_trace)
            if total_len < 100: continue
            
            # Slice MIDDLE segment to match your pairwise analysis
            start = 0
            end = total_len
            if total_len > MAX_FRAMES:
                start = (total_len - MAX_FRAMES) // 2
                end = start + MAX_FRAMES
                
            data_dict[r] = clean_trace[start:end]
            valid_regs.append(r)
            
        # 3. Build DataFrame and run LiNGAM
        df_multi = pd.DataFrame(data_dict)
        
        # Drop any remaining NaNs to ensure LiNGAM doesn't crash
        df_multi = df_multi.dropna() 
        
        res = varlingam.run_varlingam(df_multi, lags=LAG)
        
        # Extract the lag-0 (instantaneous) adjacency matrix
        adj = res['adjacency_matrices'][0] 
        
        lingam_results.append({
            "session": s.session_id,
            "regions": valid_regs,
            "adj_matrix": adj
        })

    except Exception as e:
        # print(f"Skipping session {s.session_id} due to error: {e}")
        pass

# ==========================================
# Phase 5: Aggregate and Interpret LiNGAM (Signed)
# ==========================================
if not lingam_results:
    print("No LiNGAM results generated.")
else:
    print("\n--- Aggregated LiNGAM Adjacency (Instantaneous Causality) ---")
    
    edge_list = []
    
    for res in lingam_results:
        regs = res["regions"]
        adj = res["adj_matrix"]
        
        for i, target in enumerate(regs):
            for j, source in enumerate(regs):
                if i == j: continue 
                
                weight = adj[i, j]
                if weight != 0: 
                    edge_list.append({
                        "source": source,
                        "target": target,
                        "raw_weight": weight # Keep the positive/negative sign!
                    })
                    
    df_edges = pd.DataFrame(edge_list)
    
    if not df_edges.empty:
        # 1. Calculate the mean of the raw (signed) weights
        agg_lingam = df_edges.groupby(["source", "target"])["raw_weight"].mean().reset_index()
        
        # 2. Create a temporary column of absolute values just for sorting
        agg_lingam["abs_sort"] = agg_lingam["raw_weight"].abs()
        
        # 3. Sort by the absolute magnitude, then drop the temporary column
        agg_lingam = agg_lingam.sort_values("abs_sort", ascending=False).drop(columns=["abs_sort"]).reset_index(drop=True)
        
        # Rename column for clarity in the printout
        agg_lingam.rename(columns={"raw_weight": "causal_weight_signed"}, inplace=True)
        
        print(agg_lingam.to_string(index=False))
        print("\nNote: Positive weights = Excitatory drive. Negative weights = Inhibitory drive.")
    else:
        print("LiNGAM found no distinct causal edges (all weights were 0).")

Running LiNGAM across sessions...


100%|██████████| 12/12 [01:03<00:00,  5.27s/it]


--- Aggregated LiNGAM Adjacency (Instantaneous Causality) ---
  source  target  causal_weight_signed
  DRN_NE NAc_5HT              0.280808
rNAc_5HT lNAc_DA              0.186080
  VTA_NE  NAc_DA              0.060529
lNAc_5HT rNAc_DA              0.046197
  NAc_DA  VTA_NE             -0.037499
 NAc_5HT  DRN_NE              0.001681

Note: Positive weights = Excitatory drive. Negative weights = Inhibitory drive.


- include DRN_NE -> NAc_5HT
- look into how you can get differnt lingam signs but the same pearson_r
